# Notebook 01 — Simulation Engine

**Purpose:** Generate a realistic work day for a student in a simulated job role.

Given a career role and a simulation day number, this pipeline generates:
- A briefing message from the AI "manager"
- 1-3 tasks with titles, descriptions, requirements, and deadlines
- A client email or Slack message with full context

**AI Model:** Groq — `llama-3.3-70b-versatile` via `langchain-groq`

**Key Techniques:**
- `StateGraph` (LangGraph) for the simulation workflow
- `ChatGroq` (LangChain) for structured LLM calls with JSON mode
- Typed `SimulationState` for conversation continuity across days

**Inputs:** `{ role, day, previous_context, difficulty }`

**Output:** Structured JSON with `manager_message` and `tasks` array

---
## 1. Setup & Dependencies

In [ ]:
# If running for the first time, uncomment:
# %pip install -r requirements.txt

In [ ]:
import os
import json
import time
from datetime import datetime
from typing import TypedDict, Optional
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, END

load_dotenv()

print("All imports loaded successfully")
print(f"Groq API key present: {bool(os.getenv('GROQ_API_KEY'))}")

In [ ]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in .env file")

MODEL_NAME = "llama-3.3-70b-versatile"

model = ChatGroq(
    model=MODEL_NAME,
    temperature=0.7,
    max_tokens=2000,
    api_key=GROQ_API_KEY,
)

print(f"ChatGroq initialized with model: {MODEL_NAME}")

---
## 2. Define Simulation State (LangGraph)

In [ ]:
class SimulationState(TypedDict):
    role: str
    day: int
    difficulty: str
    previous_context: Optional[str]
    system_prompt: str
    user_message: str
    result: Optional[dict]
    error: Optional[str]
    response_time: Optional[float]
    generated_at: Optional[str]

print("SimulationState defined")

---
## 3. Load Career Knowledge & Task Templates

In [ ]:
DATA_DIR = os.path.join(os.getcwd(), "data")

def load_json(filepath):
    full_path = os.path.join(DATA_DIR, filepath)
    if not os.path.exists(full_path):
        print(f"Warning: {full_path} not found.")
        return []
    with open(full_path, "r", encoding="utf-8") as f:
        return json.load(f)

mern_tasks = load_json("task_templates/mern_tasks.json")
mern_knowledge = load_json("career_knowledge/mern_developer.json")

print(f"Loaded {len(mern_tasks)} task templates")
print(f"Loaded career knowledge for: {mern_knowledge.get('role', 'unknown')}")

---
## 4. Build LangGraph Nodes

In [ ]:
TASK_EXAMPLES = json.dumps(mern_tasks[:3] if mern_tasks else [], indent=2)

def build_system_prompt(role):
    knowledge = mern_knowledge
    day_to_day = knowledge.get("day_to_day", [])
    tech_stack = knowledge.get("tech_stack", {})
    common_issues = knowledge.get("common_issues", [])
    industry_standards = knowledge.get("industry_standards", [])

    parts = [
        "You are a senior engineering manager at a mid-size tech company.",
        "You have 10+ years of experience. Professional but warm.",
        "",
        "The student is a " + role + ".",
        "",
        "### Day-to-Day Responsibilities",
    ]
    for item in day_to_day:
        parts.append("- " + item)

    parts.extend(["", "### Tech Stack"])
    for cat, tools in tech_stack.items():
        parts.append("- " + cat + ": " + ", ".join(tools))

    parts.extend(["", "### Common Issues"])
    for issue in common_issues:
        if isinstance(issue, dict):
            parts.append("- " + issue.get("issue", json.dumps(issue)))
        else:
            parts.append("- " + str(issue))

    parts.extend(["", "### Industry Standards"])
    for std in industry_standards:
        parts.append("- " + std)

    parts.extend([
        "",
        "### Task Reference Examples",
        TASK_EXAMPLES,
        "",
        "### Output Format",
        "You MUST respond with valid JSON only. No markdown, no explanation.",
        'The JSON must use this structure:',
        '{',
        '  "manager_message": "Morning briefing (2-3 paragraphs, professional but warm tone)",',
        '  "tasks": [',
        '    {',
        '      "task_id": "unique ID like task_001",',
        '      "title": "Short task title",',
        '      "description": "Detailed description (2-3 sentences)",',
        '      "task_type": "coding | design | report | meeting | bug_fix",',
        '      "difficulty": "easy | medium | hard",',
        '      "deadline_hours": 4,',
        '      "client_context": "Email or Slack message from client/stakeholder",',
        '      "expected_deliverable": "What the student needs to submit"',
        '    }',
        '  ]',
        '}',
        "",
        "### Guidelines",
        "- Generate 1-3 tasks per day",
        "- Tasks must feel realistic and role-specific",
        "- Mix task types across days (not all coding, not all meetings)",
        "- Client context should feel like a real stakeholder message",
        "- Manager message should reference the overall project narrative",
        "- Increase complexity as the day number increases",
        "- Never repeat the same task idea across different days",
    ])
    return "\n".join(parts)

def build_user_message(role, day, difficulty, previous_context=None):
    parts = [
        "Generate a work day for a " + role + ".",
        "",
        "Day: " + str(day),
        "Difficulty: " + difficulty,
        "",
    ]
    if previous_context:
        parts.extend([
            "Previous Context:",
            previous_context,
            "",
            "Reference ongoing projects. Do NOT repeat completed tasks.",
        ])
    else:
        parts.append("This is day 1. The student is new. Welcome them and assign their first tasks.")

    parts.append("")
    parts.append("Return valid JSON only. No markdown or explanation.")
    return "\n".join(parts)

print("Prompt builder functions defined")

In [ ]:
def load_data_node(state: SimulationState) -> dict:
    return {"system_prompt": "", "user_message": ""}

def build_prompts_node(state: SimulationState) -> dict:
    sp = build_system_prompt(state["role"])
    um = build_user_message(
        state["role"], state["day"],
        state["difficulty"], state.get("previous_context")
    )
    print(f"  [build_prompts] System: {len(sp)} chars, User: {len(um)} chars")
    return {"system_prompt": sp, "user_message": um}

def call_model_node(state: SimulationState) -> dict:
    messages = [
        SystemMessage(content=state["system_prompt"]),
        HumanMessage(content=state["user_message"]),
    ]
    start = time.time()
    try:
        response = model.bind(response_format={"type": "json_object"}).invoke(messages)
        elapsed = time.time() - start
        content = response.content

        result = json.loads(content)

        if "manager_message" not in result or "tasks" not in result:
            raise ValueError("Response missing required fields")

        print(f"  [call_model] Got {len(result['tasks'])} tasks in {elapsed:.2f}s")
        return {
            "result": result,
            "response_time": round(elapsed, 2),
            "generated_at": datetime.now().isoformat(),
            "error": None,
        }
    except Exception as e:
        print(f"  [call_model] Failed: {e}")
        return {"error": str(e)}

print("Graph node functions defined")

---
## 5. Compile the LangGraph

In [ ]:
def build_simulation_graph():
    workflow = StateGraph(SimulationState)

    workflow.add_node("load_data", load_data_node)
    workflow.add_node("build_prompts", build_prompts_node)
    workflow.add_node("call_model", call_model_node)

    workflow.set_entry_point("load_data")
    workflow.add_edge("load_data", "build_prompts")
    workflow.add_edge("build_prompts", "call_model")
    workflow.add_edge("call_model", END)

    return workflow.compile()

app = build_simulation_graph()
print("LangGraph simulation engine compiled successfully")

---
## 6. Helper Function

In [ ]:
def generate_simulation_day(
    role="MERN Stack Developer",
    day=1,
    difficulty="easy",
    previous_context=None,
    max_retries=2,
):
    last_error = None
    for attempt in range(max_retries):
        initial_state = SimulationState(
            role=role,
            day=day,
            difficulty=difficulty,
            previous_context=previous_context,
            system_prompt="",
            user_message="",
            result=None,
            error=None,
            response_time=None,
            generated_at=None,
        )

        try:
            final_state = app.invoke(initial_state)

            if final_state.get("error"):
                raise RuntimeError(final_state["error"])

            result = final_state["result"]
            result["_metadata"] = {
                "role": role,
                "day": day,
                "difficulty": difficulty,
                "model": MODEL_NAME,
                "response_time_seconds": final_state.get("response_time", 0),
                "generated_at": final_state.get("generated_at", ""),
                "framework": "langchain + langgraph",
            }

            return result

        except Exception as e:
            last_error = str(e)
            print(f"Attempt {attempt + 1}/{max_retries} failed: {last_error}")

    raise RuntimeError(f"All {max_retries} attempts failed. Last error: {last_error}")

print("generate_simulation_day wrapper defined")

---
## 7. Display Helper

In [ ]:
def display_simulation_day(result):
    print("=" * 70)
    print(f"SIMULATION DAY {result['_metadata']['day']} - {result['_metadata']['role']}")
    print(f"   Difficulty: {result['_metadata']['difficulty']}")
    print("=" * 70)

    print("\nMANAGER MESSAGE")
    print("=" * 70)
    print(result["manager_message"])

    print("\n" + "=" * 70)
    print(f"TASKS ({len(result['tasks'])})")
    print("=" * 70)
    for i, task in enumerate(result["tasks"], 1):
        print(f"\n{'=' * 50}")
        print(f"Task #{i}: {task['title']}")
        print(f"  ID: {task.get('task_id', 'N/A')}")
        print(f"  Type: {task.get('task_type', 'N/A')}")
        print(f"  Difficulty: {task.get('difficulty', 'N/A')}")
        print(f"  Deadline: {task.get('deadline_hours', 'N/A')} hours")
        print(f"  Description: {task.get('description', 'N/A')}")
        print(f"  Deliverable: {task.get('expected_deliverable', 'N/A')}")
        print(f"  Client Context: {task.get('client_context', 'N/A')[:80]}...")

    print("\n" + "=" * 70)
    meta = result["_metadata"]
    print(f"Framework: {meta.get('framework', 'N/A')}")
    print(f"Model: {meta['model']}")
    print(f"Response time: {meta['response_time_seconds']}s")
    print(f"Generated at: {meta['generated_at']}")
    print("=" * 70)

print("display_simulation_day function defined")

---
## 8. Tests

### Test 1: Generate tasks for MERN Stack Developer — check realism

In [ ]:
print("=" * 70)
print("TEST 1: Generate a simulation day for MERN Stack Developer (Day 3)")
print("=" * 70)

try:
    result = generate_simulation_day(
        role="MERN Stack Developer",
        day=3,
        difficulty="intermediate",
        previous_context="Days 1-2: Student set up dev environment, cloned repo, fixed a minor CSS bug. Attended standup and met the team.",
    )
    display_simulation_day(result)

    print("\n" + "=" * 70)
    print("REALISM CHECK")
    print("=" * 70)
    print(f"Manager message length: {len(result['manager_message'])} chars")
    print(f"Number of tasks: {len(result['tasks'])}")
    for task in result["tasks"]:
        print(f"   - {task['title']} ({task['task_type']}, {task['difficulty']})")
except Exception as e:
    print(f"Test failed: {e}")

### Test 2: Generate 3 consecutive days — check tasks don't repeat

In [ ]:
print("=" * 70)
print("TEST 2: Generate 3 consecutive days for MERN Stack Developer")
print("=" * 70)

all_tasks = []
previous_context_lines = []
num_days = 3

for d in range(1, num_days + 1):
    print(f"\n{'=' * 50}")
    print(f"Generating Day {d}...")
    print(f"{'=' * 50}")

    try:
        context = "\n".join(previous_context_lines) if previous_context_lines else None
        result = generate_simulation_day(
            role="MERN Stack Developer",
            day=d,
            difficulty="easy" if d <= 2 else "intermediate",
            previous_context=context,
        )

        task_titles = [t["title"] for t in result["tasks"]]
        all_tasks.extend(task_titles)

        day_summary = f"Day {d}: Completed tasks - {'; '.join(task_titles)}."
        if "manager_message" in result:
            day_summary += f" Manager: {result['manager_message'][:100]}..."
        previous_context_lines.append(day_summary)

        print(f"Day {d} generated with {len(result['tasks'])} tasks")
        for t in result["tasks"]:
            print(f"   - [{t['task_type']}] {t['title']} ({t['difficulty']})")
        meta = result["_metadata"]
        print(f"   Time: {meta['response_time_seconds']}s")

    except Exception as e:
        print(f"Day {d} failed: {e}")
        break

unique_tasks = set(all_tasks)
print(f"\n{'=' * 70}")
print("DIVERSITY CHECK")
print(f"Total tasks generated: {len(all_tasks)}")
print(f"Unique task titles: {len(unique_tasks)}")
if len(unique_tasks) == len(all_tasks):
    print("No duplicate tasks found - good diversity!")
else:
    print(f"{len(all_tasks) - len(unique_tasks)} duplicate(s) found")

### Test 3: Edge case — Day 1 with no previous context

In [ ]:
print("=" * 70)
print("TEST 3: Day 1 - No previous context (new student onboarding)")
print("=" * 70)

try:
    result = generate_simulation_day(
        role="MERN Stack Developer",
        day=1,
        difficulty="easy",
        previous_context=None,
    )
    display_simulation_day(result)
    print("\nDay 1 generated successfully without previous context!")
except Exception as e:
    print(f"Test failed: {e}")

### Test 4: Measure average tokens used per call

In [ ]:
print("=" * 70)
print("TEST 4: Response Time & Cost Analysis")
print("=" * 70)

try:
    result = generate_simulation_day(
        role="MERN Stack Developer",
        day=5,
        difficulty="medium",
        previous_context="Days 1-4: Student completed onboarding, fixed auth bug, built a data table, set up email templates.",
    )

    meta = result["_metadata"]
    print(f"\nResponse time: {meta['response_time_seconds']}s")
    print("Note: Enable LANGCHAIN_TRACING_V2=true in .env for detailed token tracking")
except Exception as e:
    print(f"Test failed: {e}")

### Test 5: Measure average response time

In [ ]:
print("=" * 70)
print("TEST 5: Response Time Measurement")
print("=" * 70)

response_times = []
num_samples = 3

for i in range(num_samples):
    print(f"\nSample {i + 1}/{num_samples}...")
    try:
        start = time.time()
        result = generate_simulation_day(
            role="MERN Stack Developer",
            day=6 + i,
            difficulty="medium",
            previous_context=f"Previous days completed. Day {5 + i} was productive.",
        )
        elapsed = time.time() - start
        response_times.append(elapsed)
        print(f"   Time: {elapsed:.2f}s")
    except Exception as e:
        print(f"   Sample failed: {e}")

if response_times:
    print(f"\n{'=' * 70}")
    print("RESPONSE TIME SUMMARY")
    print(f"{'=' * 40}")
    print(f"  Samples:      {len(response_times)}")
    print(f"  Average:      {sum(response_times) / len(response_times):.2f}s")
    print(f"  Min:          {min(response_times):.2f}s")
    print(f"  Max:          {max(response_times):.2f}s")
    print(f"{'=' * 40}")
else:
    print("\nNo successful samples to report.")

---
## 9. Summary

### Test Results Checklist
- [ ] **Test 1:** Generate tasks — check realism
- [ ] **Test 2:** Generate 3 consecutive days — check tasks don't repeat
- [ ] **Test 3:** Edge case: day 1 (no previous context)
- [ ] **Test 4:** Measure tokens used per call (via LangSmith)
- [ ] **Test 5:** Measure average response time

### Architecture
```
StateGraph(SimulationState)
  [load_data] -> [build_prompts] -> [call_model] -> END
       |               |                 |
  Initializes       Builds System +    Calls ChatGroq
  empty state       Human messages     with JSON mode
                     from templates
```

### Key Libraries
| Library | Usage |
|---------|-------|
| `langchain-groq` | `ChatGroq` — LLM calls |
| `langchain-core` | `SystemMessage`, `HumanMessage` |
| `langgraph` | `StateGraph` — workflow engine |